# Clase 06 — Repaso pre-parcial

**Somos el equipo de People Analytics de una consultora de tecnología.**

Antes de extender una oferta formal queremos saber cuán probable es que el candidato la acepte. Con esa información, compensaciones decide si arma un paquete estándar o uno reforzado.

Vamos a entrenar un modelo que prediga aceptación a partir del perfil del candidato y cómo fue el proceso de selección.

---

**Cómo trabajamos hoy.** Dividimos el curso en **6 equipos de 5 personas**. El notebook está partido en 6 partes. Antes de cada parte sorteamos un equipo: a ese equipo le toca explicar qué pasa en esa parte. Al final, la sección "Para ver en casa" cierra con comparación de modelos y costos.

In [ ]:
import random

equipos = [f"Equipo {i}" for i in range(1, 7)]
equipos_ya_sorteados = []

def sortear_equipo():
    global equipos_ya_sorteados
    disponibles = [e for e in equipos if e not in equipos_ya_sorteados]
    if not disponibles:
        equipos_ya_sorteados = []
        disponibles = equipos
    elegido = random.choice(disponibles)
    equipos_ya_sorteados.append(elegido)
    print(f"Le toca a: {elegido}")
    return elegido

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score,
)

df = pd.read_csv("candidatos_ofertas.csv")

---

## Parte 1 — Entendiendo los datos

**Preguntas para el equipo que toque — en base a su experiencia como analistas de datos, cuéntennos:**
- ¿Qué variables tenemos? ¿Cuáles son numéricas y cuáles categóricas?
- ¿Cuántos registros y columnas hay? ¿Hay valores faltantes?
- ¿Qué rangos tienen las variables numéricas? ¿Hay algo raro en el `describe()`?
- ¿Está balanceada la clase target? ¿Qué implica eso para nuestro modelo?

In [ ]:
sortear_equipo()

In [ ]:
display(df.head())
print(f"\nDimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas\n")
df.info()

In [ ]:
df.describe()

In [ ]:
conteo_target = df["acepta_oferta"].value_counts()
proporcion_target = df["acepta_oferta"].value_counts(normalize=True).round(3)

print("Cantidad de candidatos por clase:")
print(conteo_target)
print("\nProporción:")
print(proporcion_target)

plt.figure(figsize=(7, 4))
plt.bar(["Rechaza oferta (0)", "Acepta oferta (1)"],
        [conteo_target.get(0, 0), conteo_target.get(1, 0)],
        color=["#E63946", "#2E86AB"])
plt.title("Balance de clases del target 'acepta_oferta'", fontsize=13)
plt.ylabel("Cantidad de candidatos")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---

## Parte 2 — Perfilando y visualizando los datos

**Preguntas para el equipo que toque:**
- ¿Hay valores faltantes o duplicados? ¿Y outliers? ¿Qué estrategias usarían para tratarlos?
- En los histogramas y boxplots, ¿qué distribución tienen las variables? ¿Dónde hay asimetrías o valores extremos?
- En el heatmap, ¿qué variables están más correlacionadas con el target y entre sí?
- Mirando los gráficos por categoría y el scatter 2D, ¿cuáles son las variables que mejor parecen separar las clases?

In [ ]:
sortear_equipo()

In [ ]:
print("Valores faltantes por columna:")
print(df.isnull().sum())
print(f"\nFilas duplicadas en el dataset: {df.duplicated().sum()}")

In [ ]:
variables_numericas = [
    "antiguedad_empleo_actual_meses",
    "diferencial_salarial_ofrecido_pct",
    "cantidad_entrevistas_realizadas",
    "dias_desde_primer_contacto",
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, var in zip(axes.flatten(), variables_numericas):
    ax.hist(df[var].dropna(), bins=25, color="#2E86AB", edgecolor="white", alpha=0.85)
    ax.set_title(f"Distribución de {var}", fontsize=11)
    ax.set_xlabel(var)
    ax.set_ylabel("Frecuencia")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, var in zip(axes, variables_numericas):
    bp = ax.boxplot(df[var].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor="#2E86AB", alpha=0.6),
                    medianprops=dict(color="black", linewidth=2),
                    flierprops=dict(marker="o", markerfacecolor="#E63946",
                                    markersize=5, markeredgecolor="none", alpha=0.6))
    ax.set_title(var, fontsize=10)
    ax.set_xticks([])
    ax.grid(alpha=0.3)
plt.suptitle("Boxplots de variables numéricas — los puntos rojos son outliers según IQR",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
matriz_correlacion = df[variables_numericas + ["acepta_oferta"]].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(matriz_correlacion, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(matriz_correlacion.columns)))
ax.set_yticks(range(len(matriz_correlacion.columns)))
ax.set_xticklabels(matriz_correlacion.columns, rotation=45, ha="right")
ax.set_yticklabels(matriz_correlacion.columns)

for i in range(len(matriz_correlacion.columns)):
    for j in range(len(matriz_correlacion.columns)):
        ax.text(j, i, f"{matriz_correlacion.iloc[i, j]:.2f}",
                ha="center", va="center",
                color="white" if abs(matriz_correlacion.iloc[i, j]) > 0.4 else "black",
                fontsize=10)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.title("Matriz de correlación entre variables numéricas y target", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
tasa_por_seniority = df.groupby("seniority")["acepta_oferta"].mean().sort_values(ascending=False)
tasa_por_modalidad = df.groupby("modalidad_trabajo")["acepta_oferta"].mean().sort_values(ascending=False)

print("Tasa de aceptación por seniority:")
print(tasa_por_seniority.round(3))
print("\nTasa de aceptación por modalidad de trabajo:")
print(tasa_por_modalidad.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(tasa_por_seniority.index, tasa_por_seniority.values,
            color="#2E86AB", alpha=0.85, edgecolor="white")
axes[0].set_title("Tasa de aceptación por seniority", fontsize=11)
axes[0].set_ylabel("Proporción que acepta")
axes[0].set_ylim(0, 1)
axes[0].axhline(0.6, color="gray", linestyle="--", alpha=0.6, label="Promedio global (0.60)")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(tasa_por_modalidad.index, tasa_por_modalidad.values,
            color="#2E86AB", alpha=0.85, edgecolor="white")
axes[1].set_title("Tasa de aceptación por modalidad de trabajo", fontsize=11)
axes[1].set_ylabel("Proporción que acepta")
axes[1].set_ylim(0, 1)
axes[1].axhline(0.6, color="gray", linestyle="--", alpha=0.6, label="Promedio global (0.60)")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

rechaza = df[df["acepta_oferta"] == 0]
acepta = df[df["acepta_oferta"] == 1]

plt.scatter(rechaza["diferencial_salarial_ofrecido_pct"],
            rechaza["antiguedad_empleo_actual_meses"],
            c="#E63946", label="Rechaza oferta", alpha=0.55, s=35,
            edgecolors="white", linewidth=0.4)
plt.scatter(acepta["diferencial_salarial_ofrecido_pct"],
            acepta["antiguedad_empleo_actual_meses"],
            c="#2E86AB", label="Acepta oferta", alpha=0.55, s=35,
            edgecolors="white", linewidth=0.4)

plt.title("Aceptación de oferta según diferencial salarial y antigüedad en empleo actual",
          fontsize=12)
plt.xlabel("Diferencial salarial ofrecido (%)")
plt.ylabel("Antigüedad en empleo actual (meses)")
plt.legend(loc="upper right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## Parte 3 — Limpiando y preparando los datos para modelar

**Preguntas para el equipo que toque:**
- ¿Cuántos outliers se detectaron y cómo se trataron? ¿Qué otras estrategias se podrían haber usado (eliminar, winsorizing, dejarlos)?
- Los nulos se imputaron con la mediana **por grupo de seniority**, no con la mediana global. ¿Por qué tiene más sentido hacerlo así?
- ¿Por qué aparecen columnas nuevas después del `get_dummies`? ¿Qué representa cada una?
- ¿Para qué sirve `stratify=y` en el `train_test_split`? ¿Y por qué escalamos **después** del split?

In [ ]:
sortear_equipo()

In [ ]:
print("=== DETECCIÓN DE PROBLEMAS ===\n")

print(f"Nulos totales: {df.isnull().sum().sum()}")
print(f"Duplicados: {df.duplicated().sum()}\n")

print("Outliers por variable numérica (método IQR):")
for var in variables_numericas:
    Q1 = df[var].quantile(0.25)
    Q3 = df[var].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    outliers = df[(df[var] < limite_inferior) | (df[var] > limite_superior)]
    print(f"  {var}: {len(outliers)} outliers  (fuera del rango [{limite_inferior:.1f}, {limite_superior:.1f}])")

In [ ]:
df_limpio = df.copy()

# 1) Tratamiento de outliers: capar al percentil 99 (winsorizing)
for var in ["antiguedad_empleo_actual_meses", "dias_desde_primer_contacto"]:
    limite = df_limpio[var].quantile(0.99)
    df_limpio[var] = df_limpio[var].clip(upper=limite)

# 2) Imputación de nulos: mediana POR GRUPO de seniority
print("Mediana de entrevistas por seniority (usada para imputar):")
print(df_limpio.groupby("seniority")["cantidad_entrevistas_realizadas"].median())

df_limpio["cantidad_entrevistas_realizadas"] = (
    df_limpio.groupby("seniority")["cantidad_entrevistas_realizadas"]
    .transform(lambda x: x.fillna(x.median()))
)

# 3) Eliminación de duplicados
filas_antes = df_limpio.shape[0]
df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)

print(f"\n=== DESPUÉS DE LIMPIAR ===")
print(f"Filas: {df_limpio.shape[0]}  (antes eran {filas_antes})")
print(f"Nulos restantes: {df_limpio.isnull().sum().sum()}")
print(f"Duplicados restantes: {df_limpio.duplicated().sum()}")

In [ ]:
df_encoded = pd.get_dummies(df_limpio,
                             columns=["seniority", "modalidad_trabajo"],
                             drop_first=True)

print(f"Columnas originales: {df_limpio.shape[1]}")
print(f"Columnas después del encoding: {df_encoded.shape[1]}")
print(f"\nColumnas nuevas que aparecieron:")
print([c for c in df_encoded.columns if c not in df_limpio.columns])

df_encoded.head()

In [ ]:
X = df_encoded.drop(columns=["acepta_oferta"])
y = df_encoded["acepta_oferta"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

print(f"Train: {X_train.shape[0]} filas  ({y_train.mean():.1%} aceptan)")
print(f"Test:  {X_test.shape[0]} filas  ({y_test.mean():.1%} aceptan)")

In [ ]:
scaler = StandardScaler()
X_train_escalado = scaler.fit_transform(X_train)
X_test_escalado = scaler.transform(X_test)

print("Antes de escalar (primeras 3 filas de X_train):")
display(X_train.head(3).round(2))

print("\nDespués de escalar (primeras 3 filas de X_train):")
display(pd.DataFrame(X_train_escalado, columns=X_train.columns).head(3).round(2))

---

## Parte 4 — Modelando y primera evaluación

**Preguntas para el equipo que toque:**
- ¿Qué significa que `n_neighbors=5`? ¿Cómo toma KNN la decisión para un candidato nuevo?
- ¿Qué están comparando con las predicciones vs los valores reales?
- Miren la accuracy que dio. ¿Les parece buena? ¿Se animan a recomendar este modelo con solo este número?
- Tengan en cuenta el balance de clases que vimos en la Parte 1: ¿cambia eso su juicio sobre la accuracy?

In [ ]:
sortear_equipo()

In [ ]:
modelo_knn = KNeighborsClassifier(n_neighbors=5)
modelo_knn.fit(X_train_escalado, y_train)

y_pred = modelo_knn.predict(X_test_escalado)

print(f"Primeras 20 predicciones del modelo: {y_pred[:20]}")
print(f"Primeras 20 etiquetas reales:       {y_test.values[:20]}")

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy del modelo KNN con k=5: {accuracy:.2%}")

---

## Parte 5 — Evaluación más profunda: matriz de confusión y métricas por clase

**Preguntas para el equipo que toque:**
- Lean la matriz de confusión. ¿Qué son TP, TN, FP y FN en nuestro problema concreto (acepta o rechaza una oferta)?
- ¿Dónde se equivoca más el modelo, en los que aceptan o en los que rechazan?
- ¿Qué diferencia hay entre precision y recall? ¿En cuál clase le va mejor al modelo?
- Para nuestro caso de negocio, ¿les preocupa más un falso positivo (predecir que acepta y rechaza) o un falso negativo (predecir que rechaza y acepta)? ¿Por qué?

In [ ]:
sortear_equipo()

In [ ]:
matriz_confusion = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz_confusion,
    display_labels=["Rechaza", "Acepta"],
)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Matriz de confusión — KNN (k=5)", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Verdaderos Negativos (TN): {matriz_confusion[0,0]}")
print(f"Falsos Positivos (FP):     {matriz_confusion[0,1]}")
print(f"Falsos Negativos (FN):     {matriz_confusion[1,0]}")
print(f"Verdaderos Positivos (TP): {matriz_confusion[1,1]}")

In [ ]:
reporte = classification_report(
    y_test, y_pred,
    target_names=["Rechaza oferta", "Acepta oferta"],
)
print(reporte)

---

## Parte 6 — Validación y robustez del modelo

**Preguntas para el equipo que toque:**
- Miren la curva de k. ¿Qué pasa con k=1? ¿Cómo se llama ese fenómeno? ¿Por qué ocurre?
- ¿Qué pasa con k muy grandes (por ejemplo k=20)? ¿Cuál sería el problema?
- ¿Cuál k elegirían y por qué?
- El cross-validation hace 5 entrenamientos distintos y mide accuracy en cada uno. ¿Qué nos está diciendo que los 5 valores sean parecidos?
- ¿Qué significa un AUC cercano a 1? ¿Y cercano a 0.5?

In [ ]:
sortear_equipo()

In [ ]:
valores_k = range(1, 21)
accuracy_train_por_k = []
accuracy_test_por_k = []

for k in valores_k:
    modelo = KNeighborsClassifier(n_neighbors=k)
    modelo.fit(X_train_escalado, y_train)
    accuracy_train_por_k.append(modelo.score(X_train_escalado, y_train))
    accuracy_test_por_k.append(modelo.score(X_test_escalado, y_test))

plt.figure(figsize=(10, 5))
plt.plot(list(valores_k), accuracy_train_por_k,
         marker="o", linestyle="--", color="gray",
         label="Accuracy en TRAIN", alpha=0.8)
plt.plot(list(valores_k), accuracy_test_por_k,
         marker="o", linestyle="-", color="#2E86AB",
         label="Accuracy en TEST", linewidth=2)
plt.title("Accuracy de train vs test según el valor de k", fontsize=13)
plt.xlabel("Valor de k (número de vecinos)")
plt.ylabel("Accuracy")
plt.xticks(list(valores_k))
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
modelo_cv = KNeighborsClassifier(n_neighbors=5)
scores_cv = cross_val_score(modelo_cv, X_train_escalado, y_train,
                            cv=5, scoring="accuracy")

print(f"Accuracy en cada uno de los 5 folds: {np.round(scores_cv, 3)}")
print(f"Accuracy promedio:  {scores_cv.mean():.2%}")
print(f"Desvío estándar:    {scores_cv.std():.2%}")

In [ ]:
y_pred_proba = modelo_knn.predict_proba(X_test_escalado)[:, 1]
fpr, tpr, umbrales = roc_curve(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color="#2E86AB", linewidth=2,
         label=f"KNN k=5 (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--",
         label="Clasificador aleatorio (AUC = 0.5)")
plt.title("Curva ROC — KNN (k=5)", fontsize=13)
plt.xlabel("Tasa de falsos positivos (FPR)")
plt.ylabel("Tasa de verdaderos positivos (TPR)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## Para ver en casa

Lo que sigue no lo vamos a correr en clase pero es **material de parcial**. Tres cierres importantes:

1. **Matriz de costos**: traducir errores del modelo a pesos argentinos y pensar el costo económico del modelo.
2. **Comparación con Regresión Logística**: el mismo pipeline, otro modelo. ¿Cuál gana? ¿Por qué?
3. **¿Por qué Regresión Lineal NO sirve para clasificación?**: motivación conceptual para usar Logística.

In [ ]:
costo_falso_positivo = 150_000
costo_falso_negativo = 400_000

tn, fp, fn, tp = matriz_confusion.ravel()

costo_fp_total = fp * costo_falso_positivo
costo_fn_total = fn * costo_falso_negativo
costo_total = costo_fp_total + costo_fn_total

print(f"Costo de cada Falso Positivo (predije acepta, rechazó):  ${costo_falso_positivo:,}")
print(f"Costo de cada Falso Negativo (predije rechaza, aceptó):  ${costo_falso_negativo:,}")
print()
print(f"Cantidad de FP en test: {fp}  ->  costo total: ${costo_fp_total:,}")
print(f"Cantidad de FN en test: {fn}  ->  costo total: ${costo_fn_total:,}")
print()
print(f"COSTO TOTAL del modelo en el set de test: ${costo_total:,}")

In [ ]:
modelo_logistica = LogisticRegression(max_iter=1000, random_state=42)
modelo_logistica.fit(X_train_escalado, y_train)

y_pred_logistica = modelo_logistica.predict(X_test_escalado)
accuracy_logistica = accuracy_score(y_test, y_pred_logistica)

print(f"Accuracy Regresión Logística:  {accuracy_logistica:.2%}")
print(f"Accuracy KNN (k=5):            {accuracy:.2%}")

coeficientes = pd.DataFrame({
    "variable": X_train.columns,
    "coeficiente": modelo_logistica.coef_[0].round(3),
}).sort_values("coeficiente", key=abs, ascending=False)

print("\nCoeficientes de la Regresión Logística (ordenados por magnitud):")
print(coeficientes.to_string(index=False))

In [ ]:
modelo_lineal = LinearRegression()
modelo_lineal.fit(X_train_escalado, y_train)

y_pred_lineal = modelo_lineal.predict(X_test_escalado)

print(f"Rango de predicciones de Regresión Lineal: "
      f"[{y_pred_lineal.min():.3f}, {y_pred_lineal.max():.3f}]")
print(f"Predicciones fuera del rango [0, 1]: "
      f"{((y_pred_lineal < 0) | (y_pred_lineal > 1)).sum()} de {len(y_pred_lineal)}")

print("\nPor qué Regresión Lineal NO es adecuada para clasificación binaria:")
print("  - Las predicciones pueden ser negativas o mayores a 1")
print("  - No modela probabilidades")
print("  - Regresión Logística resuelve esto con la función sigmoide")